# 02 — Data Cleaning

Reads raw per-category parquet files from `data/category_cache/`, cleans text, maps Amazon categories to UNSPSC segments, and produces train/test splits.

**Inputs:** `data/category_cache/*.parquet` (from `01_data_download.ipynb`)

**Outputs:**
- `data/train.parquet`
- `data/test.parquet`
- `models/label_encoder.pkl`

**Memory-safe:** reads one row-group at a time, never loads a full file into RAM.

In [ ]:
import os, re, time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

PROJ      = '/home/tayana-gpu/ML/project_01_product_classifier'
CACHE_DIR = f'{PROJ}/data/category_cache'

In [ ]:
# Amazon category → UNSPSC Segment mapping
CATEGORY_TO_UNSPSC = {
    'raw_meta_All_Beauty':                  'Pharmaceuticals and Healthcare Products',
    'raw_meta_Amazon_Fashion':              'Apparel and Luggage and Personal Care Products',
    'raw_meta_Appliances':                  'Domestic Appliances and Supplies and Consumer Electronic Products',
    'raw_meta_Arts_Crafts_and_Sewing':      'Office Equipment and Accessories and Supplies',
    'raw_meta_Automotive':                  'Transportation and Storage and Mail Services',
    'raw_meta_Baby_Products':               'Pharmaceuticals and Healthcare Products',
    'raw_meta_Beauty_and_Personal_Care':    'Pharmaceuticals and Healthcare Products',
    'raw_meta_Books':                       'Printed Media',
    'raw_meta_CDs_and_Vinyl':               'Audio and Visual Presentation and Composing Equipment',
    'raw_meta_Cell_Phones_and_Accessories': 'Electronic Components and Supplies',
    'raw_meta_Clothing_Shoes_and_Jewelry':  'Apparel and Luggage and Personal Care Products',
    'raw_meta_Digital_Music':               'Audio and Visual Presentation and Composing Equipment',
    'raw_meta_Electronics':                 'Electronic Components and Supplies',
    'raw_meta_Grocery_and_Gourmet_Food':    'Food Beverage and Tobacco Products',
    'raw_meta_Handmade_Products':           'Apparel and Luggage and Personal Care Products',
    'raw_meta_Health_and_Household':        'Pharmaceuticals and Healthcare Products',
    'raw_meta_Health_and_Personal_Care':    'Pharmaceuticals and Healthcare Products',
    'raw_meta_Home_and_Kitchen':            'Domestic Appliances and Supplies and Consumer Electronic Products',
    'raw_meta_Industrial_and_Scientific':   'Industrial Machinery and Equipment',
    'raw_meta_Kindle_Store':                'Printed Media',
    'raw_meta_Magazine_Subscriptions':      'Printed Media',
    'raw_meta_Movies_and_TV':               'Audio and Visual Presentation and Composing Equipment',
    'raw_meta_Musical_Instruments':         'Audio and Visual Presentation and Composing Equipment',
    'raw_meta_Office_Products':             'Office Equipment and Accessories and Supplies',
    'raw_meta_Patio_Lawn_and_Garden':       'Farming and Fishing and Forestry and Wildlife Machinery',
    'raw_meta_Pet_Supplies':                'Animals and Birds and Fish',
    'raw_meta_Software':                    'Information Technology Broadcasting and Telecommunications',
    'raw_meta_Sports_and_Outdoors':         'Sports and Recreational Equipment and Supplies',
    'raw_meta_Tools_and_Home_Improvement':  'Building and Construction Machinery and Accessories',
    'raw_meta_Toys_and_Games':              'Sports and Recreational Equipment and Supplies',
    'raw_meta_Video_Games':                 'Audio and Visual Presentation and Composing Equipment',
}

In [ ]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9\s\-\/\.]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def join_list_field(field) -> str:
    if isinstance(field, list):
        return ' '.join(str(x) for x in field if x)
    if isinstance(field, str):
        return field
    return ''

def build_text(df: pd.DataFrame) -> pd.Series:
    return (
        df['title'].apply(clean_text) + ' ' +
        df['description'].apply(join_list_field).apply(clean_text) + ' ' +
        df['features'].apply(join_list_field).apply(clean_text)
    ).str.strip()

NEEDED_COLS = ['title', 'description', 'features']

def load_sampled(filepath: str, cap: int) -> pd.DataFrame:
    """Read parquet row-group by row-group. Peak RAM = one row-group, not the whole file."""
    pf    = pq.ParquetFile(filepath)
    total = pf.metadata.num_rows

    if total <= cap:
        return pf.read(columns=NEEDED_COLS).to_pandas()

    frac   = cap / total
    chunks = []
    for i in range(pf.metadata.num_row_groups):
        rg     = pf.read_row_group(i, columns=NEEDED_COLS).to_pandas()
        n_keep = max(1, round(len(rg) * frac))
        if len(rg) > n_keep:
            rg = rg.sample(n_keep, random_state=42)
        chunks.append(rg)
        del rg

    result = pd.concat(chunks, ignore_index=True)
    del chunks
    if len(result) > cap:
        result = result.sample(cap, random_state=42)
    return result

In [ ]:
# Step 1: load, clean, and tag each category file
CAP_PER_FILE = 300_000
cache_files  = sorted(f for f in os.listdir(CACHE_DIR) if f.endswith('.parquet'))

print(f'Category files found: {len(cache_files)}')
print(f'Cap per file:         {CAP_PER_FILE:,}\n')

frames  = []
t_start = time.time()

for i, fname in enumerate(cache_files, 1):
    config_name = fname.replace('.parquet', '')
    if config_name not in CATEGORY_TO_UNSPSC:
        print(f'[{i:02d}/{len(cache_files)}] SKIP (no mapping): {config_name}')
        continue

    segment = CATEGORY_TO_UNSPSC[config_name]
    t1      = time.time()

    df_cat        = load_sampled(f'{CACHE_DIR}/{fname}', CAP_PER_FILE)
    n_raw         = pq.ParquetFile(f'{CACHE_DIR}/{fname}').metadata.num_rows
    df_cat['text']           = build_text(df_cat)
    df_cat['unspsc_segment'] = segment
    frames.append(df_cat[['text', 'unspsc_segment']])
    del df_cat

    print(f'[{i:02d}/{len(cache_files)}] {fname:<55} {n_raw:>8,} raw → {len(frames[-1]):>7,} kept  ({time.time()-t1:.1f}s)')

print(f'\nAll files done in {time.time()-t_start:.0f}s')

In [ ]:
# Step 2: combine all categories
raw = pd.concat(frames, ignore_index=True)
del frames
print(f'Combined rows: {len(raw):,}')

In [ ]:
# Step 3: quality filters
print('--- Quality filters ---')

n0  = len(raw)
raw = raw[raw['text'].str.split().str.len() >= 3].copy()
print(f'Dropped (< 3 words):        {n0 - len(raw):>10,}')

n1  = len(raw)
raw = raw.drop_duplicates(subset=['text']).copy()
print(f'Dropped (exact duplicates): {n1 - len(raw):>10,}')

counts           = raw['unspsc_segment'].value_counts()
valid_segments   = counts[counts >= 500].index
dropped_segments = counts[counts < 500].index.tolist()
raw              = raw[raw['unspsc_segment'].isin(valid_segments)].copy()
if dropped_segments:
    print(f'Dropped segments (< 500 rows): {dropped_segments}')

print(f'\nFinal rows:     {len(raw):,}')
print(f'Final segments: {raw["unspsc_segment"].nunique()}')

print('\nFinal class distribution:')
counts = raw['unspsc_segment'].value_counts()
for seg, n in counts.items():
    print(f'  {seg:<65} {n:>10,}')
print(f'\nImbalance ratio: {counts.max() / counts.min():.0f}x')

In [ ]:
# Step 4: encode labels, split, save
print('--- Encoding and splitting ---')

le           = LabelEncoder()
raw['label'] = le.fit_transform(raw['unspsc_segment'])

print('Label encoding:')
for idx, cls in enumerate(le.classes_):
    print(f'  {idx:>2}: {cls}')

train_df, test_df = train_test_split(
    raw[['text', 'unspsc_segment', 'label']],
    test_size=0.20,
    stratify=raw['label'],
    random_state=42,
)

print(f'\nTrain: {len(train_df):,} | Test: {len(test_df):,}')

train_df.to_parquet(f'{PROJ}/data/train.parquet', index=False)
test_df.to_parquet(f'{PROJ}/data/test.parquet',   index=False)
joblib.dump(le, f'{PROJ}/models/label_encoder.pkl')

print('\nSaved:')
print('  data/train.parquet')
print('  data/test.parquet')
print('  models/label_encoder.pkl')
print('\nDONE. Next: run 03_eda.ipynb')